In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TRANSFORMERS_CACHE'] = '/share/u/models/'
import sys
sys.path.append('../src')

In [2]:
import torch 
import os 
from r1helpers.wrappers import R1Prompter
from nnsight import LanguageModel
import torch
import numpy as np
from transformers import set_seed
from circuitsvis.tokens import colored_tokens


def my_set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

def find_wait_token_ids(tokenizer):
    """Finds token IDs for ' wait' and ' Wait'."""
    tokens_to_check = ["wait", "Wait", " wait", " Wait"]
    token_ids = set()
    print("Attempting to encode potential 'wait' tokens:")  # Debug
    for token_str in tokens_to_check:
        ids = tokenizer.encode(token_str, add_special_tokens=False)
        print(f"  - '{token_str}' -> IDs: {ids}")  # Debug
        if len(ids) == 1:
            token_ids.add(ids[0])
        elif len(ids) > 1:
            # This warning might be important if 'wait' isn't a single token sometimes
            print(
                f"  - Warning: Token '{token_str}' split into multiple IDs: {ids}. Adding first: {ids[0]}")
            token_ids.add(ids[0])

    if not token_ids:
        raise ValueError("Could not find token IDs for 'wait' or 'Wait'.")
    print(f"Found 'wait'/'Wait' related token IDs: {token_ids}")
    return token_ids

def pprint(new, new_plus_surr):
    start = new_plus_surr[:-len(new)]
    # Ensure tokens are at most 50 characters
    tokens_start = [start[i:i+50] for i in range(0, len(start), 50)]
    tokens_new = [new[i:i+50] for i in range(0, len(new), 50)]
    # Create opacity values for each token
    opacities_start = [0.1] * len(tokens_start)
    opacities_new = [0.9] * len(tokens_new)
    return colored_tokens(tokens_start + tokens_new, 
                         opacities_start + opacities_new)

/share/u/wendler/.conda/envs/.r1installation/lib/python3.11/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [33]:
mode2strenghts = {
    "all": [1.5, 1.25, 1, 0.75, -0.75, -1, -1.25, -1.5],
    "reactive": [5, 4, 3, 2, 1.5, 1, 0.5, -0.5, -1, -1.5, -2, -3, -4, -5]#[32, 16, 8],#-8, -16, -32]
}
modes = ["all"]#,"reactive"]#["all", "reactive"]
n_new_toks = 200
outfile = "../results/multiple_prompts_steering_l1_crosscoder.csv"
normalize = True
n_rollouts_per_prompt = 3

In [4]:
import json
#l1 crosscoder
with open("../assets/l15_examples_backtracking.json", "r") as f:
    l15_examples = json.load(f)
print(l15_examples["explanations"])
layer2featuresidcs = {15: {d["feature_id"]: f"backtracking {d['explanation']}" for d in l15_examples["explanations"]}}


[{'feature_id': 18832, 'explanation': 'no'}, {'feature_id': 18663, 'explanation': 'no'}, {'feature_id': 32732, 'explanation': 'yes'}, {'feature_id': 17615, 'explanation': 'yes'}, {'feature_id': 7510, 'explanation': 'yes'}, {'feature_id': 24996, 'explanation': 'yes'}, {'feature_id': 17455, 'explanation': 'no'}, {'feature_id': 20781, 'explanation': 'yes'}, {'feature_id': 20197, 'explanation': 'no'}, {'feature_id': 31660, 'explanation': 'no'}, {'feature_id': 14122, 'explanation': '26.3'}, {'feature_id': 10256, 'explanation': 'yes'}, {'feature_id': 9725, 'explanation': 'no'}, {'feature_id': 12819, 'explanation': 'no'}, {'feature_id': 898, 'explanation': 'no'}, {'feature_id': 4118, 'explanation': 'no'}, {'feature_id': 25474, 'explanation': 'yes'}, {'feature_id': 31444, 'explanation': '28.8'}, {'feature_id': 31673, 'explanation': 'no'}, {'feature_id': 17909, 'explanation': 'yes'}, {'feature_id': 28514, 'explanation': 'no'}, {'feature_id': 12624, 'explanation': 'no'}, {'feature_id': 21870, 'e

In [5]:
# load the cross-coder decoder matrices
base_path = "/tmp/wendler/Crosscoder-Llama-3.1-8B-vs-Llama-R1-Distill-8B/BatchTopK-Crosscoder/"
base_path = "/tmp/wendler/Crosscoder-Llama-3.1-8B-vs-Llama-R1-Distill-8B/L1-Crosscoder/"

layer2paths = {7:"L7R/cc_weights.pt", 15:"L15R/cc_weights.pt", 22:"L23R/cc_weights.pt"}
layer2features = {}
for lidx, path in layer2paths.items():
    if lidx in layer2featuresidcs:  
        d = torch.load(os.path.join(base_path, path), map_location="cpu")
        layer2features[lidx] = d["decoder.weight"][1]/d["decoder.weight"][1].norm(dim=-1, keepdim=True) if normalize else d["decoder.weight"][1]

/tmp/ipykernel_1701997/1860122577.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  d = torch.load(os.path.join(base_path, path), map_location="cpu")


In [6]:
model_name = "/share/u/models/deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
prompter = R1Prompter(model_name=model_name)
lm = LanguageModel(model_name, torch_dtype=torch.bfloat16, device_map="auto")

In [7]:
tokenizer = prompter.tokenizer
wait_toks = find_wait_token_ids(tokenizer)
print(wait_toks)


Attempting to encode potential 'wait' tokens:
  - 'wait' -> IDs: [11748]
  - 'Wait' -> IDs: [14524]
  - ' wait' -> IDs: [3868]
  - ' Wait' -> IDs: [14144]
Found 'wait'/'Wait' related token IDs: {14144, 14524, 11748, 3868}
{14144, 14524, 11748, 3868}


In [8]:
from functools import partial

@torch.no_grad()
def gen(lm, toks, n_new_toks=50, temperature=0.6, do_sample=True, seed=2):
    if seed:
        my_set_seed(seed)
    with lm.generate(toks, max_new_tokens=n_new_toks, do_sample=do_sample, temperature=temperature) as tracer:
        gen_toks = lm.generator.output.save()
    n_real_new_tokens = len(gen_toks[0]) - len(toks)
    generated_text = lm.tokenizer.decode(gen_toks[0][-n_real_new_tokens:])
    generated_text_plus_surr = lm.tokenizer.decode(gen_toks[0][-n_real_new_tokens-50:])
    return {"new_txt":  generated_text, 
                "all_toks": gen_toks.cpu().detach(), 
                "new_plus_surr": generated_text_plus_surr}

@torch.no_grad()
def steer(lm, toks, vecs, 
          from_tok_idx = -1,
          mode = "all", 
          layer_idcs=[], 
          n_new_toks=10, 
          do_sample=True, 
          alpha=0.0, 
          temperature=0.6, 
          thres=-0.01,
          verbose=False, 
          seed=2):
    assert mode in ["all", "reactive"], "mode must be either 'all' or 'reactive'"
    if from_tok_idx >= 0 and from_tok_idx < len(toks):
        print("WARNING: from_tok_idx is within the input sequence. Is this what you wanted to do?")
    if seed:
        my_set_seed(seed)
    if mode == "all":
        total = 0
        my_curr_idx = len(toks)
        @torch.no_grad()
        def steer_hook(module, input, output, from_tok_idx=None, vec=None, coef=None, thres=None):
            nonlocal total, fire_cnt, my_curr_idx
            # shape: batch x seq x dmodel 
            if my_curr_idx >= from_tok_idx or from_tok_idx == -1:
                if output[0].shape[1] > 1:
                    tok_norms = output[0][0,from_tok_idx:].norm(dim=-1, keepdim=True)
                    output[0][0,from_tok_idx:] += tok_norms*coef*vec.to(output[0].device)
                else:
                    v1 = output[0][0,0]
                    v1norm = v1.norm()
                    output[0][0,0] += v1norm*coef*vec.to(output[0].device)
            total+=1
            my_curr_idx += 1
            return output
        myhooks = [lm.model.layers[lidx].register_forward_hook( \
            partial(steer_hook, from_tok_idx=from_tok_idx, vec=vecs[idx], coef=alpha, thres=thres)) for idx, lidx in enumerate(layer_idcs)]
        # seed has already been set within steer
        try: 
            out = gen(lm, toks, n_new_toks, temperature=temperature, do_sample=do_sample, seed=None)
        finally:
            for hook in myhooks:
                hook.remove()
        return out
    elif mode == "reactive":
        fire_cnt = 0
        total = 0
        my_curr_idx = len(toks)
        @torch.no_grad()
        def steer_hook(module, input, output, from_tok_idx=None, vec=None, coef=None, thres=None):
            nonlocal total, fire_cnt, my_curr_idx
            # shape: batch x seq x dmodel 
            if my_curr_idx >= from_tok_idx or from_tok_idx == -1:
                if output[0].shape[1] > 1:
                    tok_norms = output[0][0,from_tok_idx:].norm(dim=-1, keepdim=True)
                    output[0][0,from_tok_idx:] += tok_norms*coef*vec.to(output[0].device)
                else:
                    v1 = output[0][0,0]
                    v2 = vec.to(output[0].device).to(v1.dtype)
                    v1norm = v1.norm()
                    v1/=v1.norm()
                    v2/=v2.norm()
                    proj = torch.dot(v1, v2)
                    v1 *= v1norm
                    if proj < thres:
                        fire_cnt += 1
                        output[0][0,0] += v1norm*coef*vec.to(output[0].device)
            total+=1
            my_curr_idx += 1
            return output
        myhooks = [lm.model.layers[lidx].register_forward_hook( \
            partial(steer_hook, from_tok_idx=from_tok_idx, vec=vecs[idx], coef=alpha, thres=thres)) for idx, lidx in enumerate(layer_idcs)]
        # seed has already been set within steer
        try: 
            out = gen(lm, toks, n_new_toks, temperature=temperature, do_sample=do_sample, seed=None)
            if from_tok_idx == -1:
                out["fire_fraction"] = 100*fire_cnt/(my_curr_idx-len(toks))
            else:
                out["fire_fraction"] = 100*fire_cnt/(my_curr_idx-from_tok_idx)
        finally:
            for hook in myhooks:
                hook.remove()
        return out

In [12]:
with open("../assets/wait_subsequences_from_outputs.json", "r") as f:
    wait_subsequences = json.load(f)

In [30]:
len(wait_subsequences)

622

In [50]:
# filter the sequences 
from collections import defaultdict
dataset = []
prompts_seen = set()
counts = defaultdict(int)
seeds = defaultdict(set)
for d in wait_subsequences:
    if d["config"] == "Sampling (DeepSeek recommended)":
        if counts[d["prompt"]] < n_rollouts_per_prompt and d["seed"] not in seeds[d["prompt"]]:
            toks = d["subsequence_tokens"]
            out = gen(lm, toks, n_new_toks=1)
            if "wait" in out["new_txt"].lower():
                counts[d["prompt"]] += 1
                seeds[d["prompt"]].add(d["seed"])
                dataset.append(d)
print(len(dataset))


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

12


In [51]:
d

{'original_entry_index': 227,
 'prompt': 'prompt_4',
 'seed': 41,
 'config': 'Sampling (temp=1.0)',
 'original_output_preview': "<｜User｜>What's the sum of all proper divisors of 36?<｜Assistant｜><think>\nOkay, so I need to find the...",
 'wait_token_index_in_original': 942,
 'subsequence_tokens': [128000,
  128011,
  3923,
  596,
  279,
  2694,
  315,
  682,
  6300,
  3512,
  42314,
  315,
  220,
  1927,
  30,
  128012,
  128013,
  198,
  33413,
  11,
  779,
  358,
  1205,
  311,
  1505,
  279,
  2694,
  315,
  682,
  6300,
  3512,
  42314,
  315,
  220,
  1927,
  13,
  89290,
  11,
  1095,
  757,
  1176,
  19635,
  1148,
  6300,
  3512,
  42314,
  527,
  13,
  1442,
  358,
  6227,
  12722,
  11,
  264,
  6300,
  50209,
  315,
  264,
  1396,
  374,
  264,
  50209,
  315,
  430,
  1396,
  44878,
  279,
  1396,
  5196,
  13,
  2100,
  11,
  369,
  3187,
  11,
  279,
  6300,
  3512,
  42314,
  315,
  220,
  21,
  1053,
  387,
  220,
  16,
  11,
  220,
  17,
  11,
  323,
  220,
  18,
  11,


In [52]:
from collections import defaultdict 
import pandas as pd
import os
from tqdm import tqdm

# Check if file exists and remove it to start fresh
if os.path.exists(outfile):
    os.remove(outfile)

for d in dataset:
    toks = d["subsequence_tokens"]
    wait_tok_idx = d["wait_token_index_in_original"]
    # create reference sequence
    out = gen(lm, toks, n_new_toks=n_new_toks)
    # Create a single row dataframe
    row_data = {
        "layer_idx": [-1],
        "feature_idx": [-1],
        "feature_summary": ["reference"],
        "strength": [0],
        "mode": ["gen"],
        "text_after_wait": [tokenizer.decode(out["all_toks"][0][wait_tok_idx:])],
        "full_response": [out["new_txt"]],
        "text_before_wait": [tokenizer.decode(toks[:wait_tok_idx])],
        "prompt": [d["prompt"]],
        "steering_fraction": [0]
    }
    # Convert to DataFrame
    row_df = pd.DataFrame(row_data)
    
    # Append to file (create file with header if it doesn't exist)
    row_df.to_csv(outfile, mode='a', header=not os.path.exists(outfile), index=False)
    # run interventions 
    for mode in modes:
        for layer_idx, vecs in tqdm(list(layer2features.items())):
            for fidx in layer2featuresidcs[layer_idx].keys():
                for strength in mode2strenghts[mode]:
                    assert len(toks) == wait_tok_idx, "wait_tok_idx is not the last token in the sequence"
                    out = steer(lm, toks, 
                                vecs=vecs[fidx].unsqueeze(0), 
                                layer_idcs=[layer_idx],
                                mode=mode, 
                                alpha=strength, 
                                n_new_toks=n_new_toks, 
                                from_tok_idx=-1)
                    print(f"layer_idx: {layer_idx}, feature_idx: {fidx}, strength: {strength}, mode: {mode}")
                    print(f"feature summary: {layer2featuresidcs[layer_idx][fidx]}")
                    print(tokenizer.decode(out["all_toks"][0][wait_tok_idx:wait_tok_idx+100]))
                    print("-"*100)
                    
                    # Create a single row dataframe
                    row_data = {
                        "layer_idx": [layer_idx],
                        "feature_idx": [fidx],
                        "feature_summary": [layer2featuresidcs[layer_idx][fidx]],
                        "strength": [strength],
                        "mode": [mode],
                        "text_after_wait": [tokenizer.decode(out["all_toks"][0][wait_tok_idx:])],
                        "full_response": [out["new_txt"]],
                        "text_before_wait": [tokenizer.decode(toks[:wait_tok_idx])],
                        "prompt": [d["prompt"]]
                    }
                    if "fire_fraction" in out:
                        row_data["steering_fraction"] = [out["fire_fraction"]]
                        print(f"steering fraction: {out['fire_fraction']}")
                    else:
                        row_data["steering_fraction"] = [1]
                    # Convert to DataFrame
                    row_df = pd.DataFrame(row_data)
                    
                    # Append to file (create file with header if it doesn't exist)
                    row_df.to_csv(outfile, mode='a', header=not os.path.exists(outfile), index=False)

# Read the complete dataframe from disk
df = pd.read_csv(outfile)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


  0%|          | 0/1 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


layer_idx: 15, feature_idx: 18832, strength: 1.5, mode: all
feature summary: backtracking no
 wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait I wait wait wait wait wait wait wait wait hold wait wait wait wait wait wait wait wait wait wait hold wait hold hold wait wait wait wait wait wait wait wait wait I wait then wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait let wait wait hold wait wait wait hold wait wait wait let wait I wait wait I wait
----------------------------------------------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


layer_idx: 15, feature_idx: 18832, strength: 1.25, mode: all
feature summary: backtracking no
 wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait wait let wait wait wait wait wait wait wait wait wait I wait wait wait wait wait wait wait let hold wait wait wait wait I wait maybe wait wait wait hold wait hold hold wait wait wait wait wait wait wait wait wait I wait wait wait wait wait wait wait wait hang wait wait wait wait wait wait wait wait wait I wait wait let wait let hold wait wait wait hold wait wait wait let wait I I wait I wait
----------------------------------------------------------------------------------------------------


...

layer_idx: 15, feature_idx: 20781, strength: 1.25, mode: all
feature summary: backtracking yes
WaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitAlrightWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitSoWaitWaitWaitWaitWaitWaitWaitWaitAlrightWaitWaitAlrightWaitWaitWaitOkayWaitWaitWaitSoWaitWaitAlrightWaitWaitSoWaitWaitAlrightSoWaitWaitSoWaitWaitSoWaitSoWaitWaitWaitWaitWait
----------------------------------------------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


layer_idx: 15, feature_idx: 20781, strength: 1, mode: all
feature summary: backtracking yes
WaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitAlrightWaitWaitWaitWaitWaitWaitWaitWaitWaitWaitSoWaitWaitWaitWaitWaitWaitWaitWaitAlrightWaitWaitAlrightWaitWaitWaitOkayWaitWaitWaitAlrightSoWaitAlrightWaitWaitSoWaitWaitAlrightSoWaitWaitSoAlrightWaitSoWaitSoWaitWaitWaitWaitWait
----------------------------------------------------------------------------------------------------


  0%|          | 0/1 [06:53<?, ?it/s]


KeyboardInterrupt: 